# GCF Project Evaluation Pipeline
**Evaluation Quant Specialist | Green Climate Fund**

---
**Workflow:**
1. `Section 1` — Fetch & cache API data (run once; reloads from cache)
2. `Section 2` — Data inventory (fields, completeness, schema)
3. `Section 3` — Portfolio tabulation (any cut: region, sector, status)
4. `Section 4` — Evaluation questions (plug in your question, get a table)
5. `Section 5` — FT-style visualizations
6. `Section 6` — Narrative generation


## Setup

In [ ]:
import os, json, warnings
from pathlib import Path
from datetime import datetime

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from IPython.display import display, Markdown, HTML

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)

# ── Configuration ──────────────────────────────────────────────────────────
API_URL   = 'http://api.gcfund.org/v1/projects'
API_KEY   = os.getenv('GCF_API_KEY', '')          # set env var or paste here
CACHE_DIR = Path('gcf_cache')
OUT_DIR   = Path('gcf_output')
CACHE_FILE = CACHE_DIR / 'projects_raw.parquet'
FORCE_REFRESH = False   # set True to re-fetch from API even if cache exists

CACHE_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)
print('Config OK. Cache:', CACHE_FILE)

## Section 1 — Fetch & Cache
_Run once. On subsequent sessions, loads from parquet cache instantly._

In [ ]:
def fetch_all_pages(url: str, api_key: str) -> list[dict]:
    """Paginate through the GCF projects API and return all records."""
    headers = {'Accept': 'application/json'}
    if api_key:
        headers['Authorization'] = f'Bearer {api_key}'
    records, page = [], 1
    while url:
        resp = requests.get(url, headers=headers, params={'page': page}, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        # Try common envelope keys
        items = (data.get('projects') or data.get('data') or
                 data.get('items') or data.get('results') or [])
        if not items:
            break
        records.extend(items)
        print(f'  Page {page}: +{len(items)} records (total {len(records)})')
        # Follow next-page link if present, else increment page
        next_url = data.get('next') or data.get('nextPage')
        if next_url and next_url != url:
            url, page = next_url, 1
        elif data.get('totalPages') and page < data['totalPages']:
            page += 1
        else:
            break
    return records


def normalize_df(records: list[dict]) -> pd.DataFrame:
    df = pd.json_normalize(records, sep='_')
    df.columns = [c.lower().strip() for c in df.columns]
    return df


if not FORCE_REFRESH and CACHE_FILE.exists():
    df_raw = pd.read_parquet(CACHE_FILE)
    print(f'Loaded from cache: {len(df_raw):,} projects | {df_raw.shape[1]} fields')
    print(f'Cache written: {datetime.fromctime(CACHE_FILE.stat().st_ctime):%Y-%m-%d %H:%M}')
else:
    print('Fetching from API…')
    rows = fetch_all_pages(API_URL, API_KEY)
    df_raw = normalize_df(rows)
    df_raw.to_parquet(CACHE_FILE, index=False)
    print(f'Fetched & cached: {len(df_raw):,} projects | {df_raw.shape[1]} fields')

df = df_raw.copy()

## Section 2 — Data Inventory
_Understand your fields before analysis._

In [ ]:
# Field catalogue: name, type, completeness
inventory = pd.DataFrame({
    'field': df.columns,
    'dtype': df.dtypes.values,
    'non_null': df.notna().sum().values,
    'completeness_%': (df.notna().mean() * 100).round(1).values,
    'n_unique': df.nunique().values,
    'sample': [df[c].dropna().iloc[0] if df[c].notna().any() else '' for c in df.columns]
})
display(HTML('<h4>Field Inventory</h4>'))
display(inventory.sort_values('completeness_%', ascending=False).reset_index(drop=True))

In [ ]:
# Auto-detect key columns (edit if API names differ)
def detect_col(df, *candidates):
    for c in candidates:
        if c in df.columns: return c
    return None

COL = {
    'id':       detect_col(df, 'id', 'project_id', 'fp_number'),
    'title':    detect_col(df, 'title', 'name', 'project_name'),
    'country':  detect_col(df, 'country', 'countries', 'recipient_country'),
    'region':   detect_col(df, 'region', 'geographic_region'),
    'sector':   detect_col(df, 'sector', 'theme', 'thematic_area'),
    'status':   detect_col(df, 'status', 'project_status'),
    'division': detect_col(df, 'division', 'modality'),
    'amount':   detect_col(df, 'amount', 'gcf_amount', 'approved_amount', 'commitment'),
    'date':     detect_col(df, 'approval_date', 'date_approved', 'board_approval_date'),
    'entity':   detect_col(df, 'entity', 'implementing_entity', 'accredited_entity'),
}

print('Detected columns:')
for k, v in COL.items():
    flag = '✓' if v else '✗ NOT FOUND'
    print(f'  {k:12s} → {v or "—":30s} {flag}')

# Convert amount to numeric, date to datetime
if COL['amount']:
    df[COL['amount']] = pd.to_numeric(df[COL['amount']], errors='coerce')
if COL['date']:
    df[COL['date']] = pd.to_datetime(df[COL['date']], errors='coerce')
    df['approval_year'] = df[COL['date']].dt.year

## Section 3 — Portfolio Tabulation
_Quick cross-tabs on any dimension. Edit `GROUP_BY` to slice differently._

In [ ]:
def portfolio_table(df, group_by: str, amount_col: str = None, label: str = None):
    """Summarise portfolio by any grouping column."""
    if group_by not in df.columns:
        print(f'Column {group_by!r} not found'); return
    agg = {'project_count': (COL['id'] or group_by, 'count')}
    if amount_col and amount_col in df.columns:
        agg['total_amount_M'] = (amount_col, lambda x: x.sum() / 1e6)
        agg['avg_amount_M']   = (amount_col, lambda x: x.mean() / 1e6)
        agg['share_%']        = (amount_col, lambda x: x.sum() / df[amount_col].sum() * 100)
    tbl = df.groupby(group_by).agg(**agg).round(2)
    tbl = tbl.sort_values('total_amount_M' if 'total_amount_M' in tbl else 'project_count', ascending=False)
    display(HTML(f'<h4>Portfolio by {label or group_by.title()}</h4>'))
    display(tbl)
    return tbl

# ── Run your tabulations ──
if COL['region']:  portfolio_table(df, COL['region'],  COL['amount'], 'Region')
if COL['sector']:  portfolio_table(df, COL['sector'],  COL['amount'], 'Sector/Theme')
if COL['status']:  portfolio_table(df, COL['status'],  COL['amount'], 'Project Status')
if COL['entity']:  portfolio_table(df, COL['entity'],  COL['amount'], 'Accredited Entity')

In [ ]:
# Yearly approval trend
if 'approval_year' in df.columns and COL['amount']:
    trend = df.groupby('approval_year').agg(
        projects=(COL['id'], 'count'),
        total_M=(COL['amount'], lambda x: x.sum() / 1e6)
    ).round(1)
    display(HTML('<h4>Approvals by Year</h4>'))
    display(trend)

## Section 4 — Evaluation Questions
_Template: define your question, filter, and indicators. Duplicate this cell for each question._

In [ ]:
# ══════════════════════════════════════════════════════
# EVALUATION QUESTION TEMPLATE
# Duplicate this cell and modify for each question.
# ══════════════════════════════════════════════════════

EQ_LABEL = 'EQ1: To what extent does the GCF portfolio reach the most vulnerable countries?'

# ── Filter (edit as needed) ────────────────────────────
eq_df = df.copy()
# Example: restrict to active / approved projects
# if COL['status']:
#     eq_df = eq_df[eq_df[COL['status']].isin(['Active', 'Approved'])]

# ── Indicators ────────────────────────────────────────
indicators = {}
indicators['N projects'] = len(eq_df)
if COL['country']:
    indicators['N countries'] = eq_df[COL['country']].nunique()
if COL['amount']:
    indicators['Total GCF (USD M)'] = eq_df[COL['amount']].sum() / 1e6
    indicators['Avg project size (USD M)'] = eq_df[COL['amount']].mean() / 1e6
if COL['region']:
    top_region = eq_df[COL['region']].value_counts().idxmax()
    indicators['Largest region'] = top_region

# ── Display ────────────────────────────────────────────
display(HTML(f'<h3>{EQ_LABEL}</h3>'))
ind_df = pd.DataFrame.from_dict(indicators, orient='index', columns=['Value'])
display(ind_df)

# Cross-tab for this question
if COL['region']:
    portfolio_table(eq_df, COL['region'], COL['amount'], 'Region (EQ1)')

In [ ]:
# ── EQ2: Private Sector (PSF) portfolio ───────────────
EQ_LABEL = 'EQ2: Private Sector Facility — scale, concentration, risk profile'

psf_df = df.copy()
if COL['division']:
    psf_df = psf_df[psf_df[COL['division']].str.upper().str.contains('PSF|PRIVATE', na=False)]

if psf_df.empty:
    display(HTML('<i>No PSF records found — check division/modality column values above.</i>'))
else:
    indicators_psf = {'N PSF projects': len(psf_df)}
    if COL['country']:  indicators_psf['Countries'] = psf_df[COL['country']].nunique()
    if COL['amount']:
        total = psf_df[COL['amount']].sum() / 1e6
        indicators_psf['Total PSF (USD M)'] = round(total, 1)
        indicators_psf['Concentration (top project %)'] = round(
            psf_df[COL['amount']].max() / psf_df[COL['amount']].sum() * 100, 1)
        indicators_psf['CoV (volatility proxy)'] = round(
            psf_df[COL['amount']].std() / psf_df[COL['amount']].mean(), 3)

    display(HTML(f'<h3>{EQ_LABEL}</h3>'))
    display(pd.DataFrame.from_dict(indicators_psf, orient='index', columns=['Value']))
    if COL['country']: portfolio_table(psf_df, COL['country'], COL['amount'], 'Country (PSF)')

## Section 5 — FT-Style Visualizations
_Financial Times chart aesthetic: salmon background, clean sans-serif, minimal grid._

In [ ]:
# ── FT Theme ──────────────────────────────────────────────────────────────
FT = {
    'bg':      '#FFF1E5',   # FT signature pink/salmon
    'blue':    '#0F5499',   # FT deep blue
    'red':     '#990F3D',   # FT claret
    'orange':  '#FF8833',   # FT orange
    'teal':    '#0D7680',   # FT teal
    'grey':    '#66605C',   # FT warm grey
    'grid':    '#E0D5C9',   # FT light grid
    'font':    'Arial',
}
FT_PALETTE = [FT['blue'], FT['red'], FT['orange'], FT['teal'], '#593380', '#006F9B']

def ft_base(figsize=(10, 5.5), title='', subtitle='', source='GCF Project API'):
    """Create an FT-style figure. Returns (fig, ax)."""
    fig, ax = plt.subplots(figsize=figsize, facecolor=FT['bg'])
    ax.set_facecolor(FT['bg'])
    # Spines
    for sp in ['top', 'right', 'left']:
        ax.spines[sp].set_visible(False)
    ax.spines['bottom'].set_color(FT['grey'])
    # Grid
    ax.yaxis.grid(True, color=FT['grid'], linewidth=0.6, zorder=0)
    ax.xaxis.grid(False)
    ax.tick_params(colors=FT['grey'], labelsize=9)
    # Title
    if title:
        fig.text(0.02, 0.97, title, ha='left', va='top',
                 fontsize=13, fontweight='bold', color='#1A1A1A', fontfamily=FT['font'])
    if subtitle:
        fig.text(0.02, 0.91, subtitle, ha='left', va='top',
                 fontsize=9.5, color=FT['grey'], fontfamily=FT['font'])
    if source:
        fig.text(0.02, 0.01, f'Source: {source}', ha='left', va='bottom',
                 fontsize=7.5, color=FT['grey'], fontfamily=FT['font'])
    fig.subplots_adjust(top=0.82, bottom=0.12)
    return fig, ax

print('FT theme loaded. Use ft_base() to start any chart.')

In [ ]:
# ── Chart 1: Horizontal bar — top countries by GCF commitment ─────────────
if COL['country'] and COL['amount']:
    top_n = 12
    top = (df.groupby(COL['country'])[COL['amount']]
             .sum()
             .nlargest(top_n)
             .sort_values()
             / 1e6)

    fig, ax = ft_base(
        figsize=(9, top_n * 0.45 + 1.5),
        title=f'Top {top_n} Recipients by GCF Commitment',
        subtitle='USD millions'
    )
    bars = ax.barh(top.index, top.values, color=FT['blue'], height=0.6, zorder=3)
    # Value labels
    for bar, val in zip(bars, top.values):
        ax.text(val + top.max() * 0.01, bar.get_y() + bar.get_height() / 2,
                f'${val:,.0f}M', va='center', ha='left',
                fontsize=8, color=FT['grey'], fontfamily=FT['font'])
    ax.set_xlabel('USD millions', color=FT['grey'], fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))
    ax.yaxis.grid(False)
    ax.xaxis.grid(True, color=FT['grid'], linewidth=0.6, zorder=0)
    plt.tight_layout(rect=[0, 0.04, 1, 0.88])
    plt.savefig(OUT_DIR / 'chart_top_countries.png', dpi=220, bbox_inches='tight', facecolor=FT['bg'])
    plt.show()

In [ ]:
# ── Chart 2: Grouped bar — approvals by year and sector ───────────────────
if 'approval_year' in df.columns and COL['sector'] and COL['amount']:
    pivot = (df.groupby(['approval_year', COL['sector']])[COL['amount']]
               .sum().unstack(fill_value=0) / 1e6)
    # Keep top 4 sectors
    top_sectors = pivot.sum().nlargest(4).index
    pivot = pivot[top_sectors]

    fig, ax = ft_base(
        figsize=(11, 5.5),
        title='GCF Approvals by Year and Sector',
        subtitle='USD millions'
    )
    x = np.arange(len(pivot))
    w = 0.18
    for i, (col, color) in enumerate(zip(pivot.columns, FT_PALETTE)):
        ax.bar(x + i * w - (len(top_sectors) - 1) * w / 2,
               pivot[col], width=w, label=col, color=color, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index.astype(int), fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))
    ax.legend(frameon=False, fontsize=8, loc='upper left',
              labelcolor=FT['grey'], ncol=2)
    plt.tight_layout(rect=[0, 0.04, 1, 0.88])
    plt.savefig(OUT_DIR / 'chart_yearly_by_sector.png', dpi=220, bbox_inches='tight', facecolor=FT['bg'])
    plt.show()

In [ ]:
# ── Chart 3: Small multiples — status distribution by region ──────────────
if COL['region'] and COL['status']:
    pivot = (df.groupby([COL['region'], COL['status']]).size()
               .unstack(fill_value=0))
    regions = pivot.sum(axis=1).nlargest(6).index
    pivot = pivot.loc[regions]

    fig, axes = plt.subplots(2, 3, figsize=(13, 7), facecolor=FT['bg'])
    fig.text(0.02, 0.97, 'Project Status by Region', ha='left', va='top',
             fontsize=13, fontweight='bold', fontfamily=FT['font'])
    fig.text(0.02, 0.01, 'Source: GCF Project API', ha='left', va='bottom',
             fontsize=7.5, color=FT['grey'], fontfamily=FT['font'])

    for ax, region in zip(axes.flat, pivot.index):
        row = pivot.loc[region].sort_values(ascending=False)[:5]
        ax.set_facecolor(FT['bg'])
        bars = ax.bar(range(len(row)), row.values,
                      color=FT_PALETTE[:len(row)], zorder=3)
        ax.set_xticks(range(len(row)))
        ax.set_xticklabels(row.index, fontsize=7, rotation=30, ha='right', color=FT['grey'])
        ax.set_title(region, fontsize=9, fontweight='bold', color='#1A1A1A')
        ax.yaxis.grid(True, color=FT['grid'], linewidth=0.5)
        for sp in ['top', 'right', 'left']: ax.spines[sp].set_visible(False)
        ax.spines['bottom'].set_color(FT['grey'])
        ax.tick_params(colors=FT['grey'])

    fig.tight_layout(rect=[0, 0.04, 1, 0.93])
    plt.savefig(OUT_DIR / 'chart_status_by_region.png', dpi=220, bbox_inches='tight', facecolor=FT['bg'])
    plt.show()

## Section 6 — Narrative Generation
_Auto-drafts a findings summary from your indicators. Edit the template for your report style._

In [ ]:
def build_narrative(df, col, label='Portfolio'):
    """
    Generate a structured text narrative from the dataframe.
    Returns markdown string.
    """
    lines = [f'## {label} — Key Findings\n', f'*Generated: {datetime.now():%Y-%m-%d %H:%M}*\n']
    n = len(df)
    lines.append(f'\n### Portfolio Scale')
    lines.append(f'The {label.lower()} comprises **{n:,} projects**')

    if col['country'] and col['country'] in df.columns:
        nc = df[col['country']].nunique()
        lines[-1] += f' across **{nc} countries/territories**'
    lines[-1] += '.'

    if col['amount'] and col['amount'] in df.columns:
        total_m = df[col['amount']].sum() / 1e6
        avg_m   = df[col['amount']].mean() / 1e6
        med_m   = df[col['amount']].median() / 1e6
        largest = df[col['amount']].max() / 1e6
        lines.append(f'\n### Financial Profile')
        lines.append(f'Total GCF commitment stands at **USD {total_m:,.1f} million**, '
                     f'with a mean project size of USD {avg_m:,.1f}M (median: USD {med_m:,.1f}M). '
                     f'The largest single project is USD {largest:,.1f}M, '
                     f'representing {largest/total_m*100:.1f}% of total portfolio exposure.')

    if col['region'] and col['region'] in df.columns:
        by_region = df.groupby(col['region']).size().sort_values(ascending=False)
        top_r, top_n = by_region.index[0], by_region.iloc[0]
        lines.append(f'\n### Geographic Distribution')
        lines.append(f'The largest regional concentration is **{top_r}** with {top_n} projects '
                     f'({top_n/n*100:.1f}% of portfolio count).')

    if col['status'] and col['status'] in df.columns:
        by_status = df[col['status']].value_counts()
        lines.append(f'\n### Implementation Status')
        status_text = ', '.join([f'{s}: {v} ({v/n*100:.0f}%)' for s, v in by_status.items()])
        lines.append(f'Project status breakdown: {status_text}.')

    lines.append('\n---\n*Narrative auto-generated. Edit and expand with evaluation judgements.*')
    return '\n'.join(lines)


narrative_md = build_narrative(df, COL, label='GCF Full Portfolio')
display(Markdown(narrative_md))

# Save to file
report_path = OUT_DIR / f'gcf_narrative_{datetime.now():%Y%m%d}.md'
report_path.write_text(narrative_md, encoding='utf-8')
print(f'Saved: {report_path}')

In [ ]:
# ── Export full processed data ─────────────────────────────────────────────
export_path = OUT_DIR / f'gcf_projects_{datetime.now():%Y%m%d}.xlsx'
with pd.ExcelWriter(export_path, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='All Projects', index=False)
    if COL['region']:
        df.groupby(COL['region']).size().to_frame('count').to_excel(writer, sheet_name='By Region')
    if COL['sector']:
        df.groupby(COL['sector']).size().to_frame('count').to_excel(writer, sheet_name='By Sector')
print(f'Exported: {export_path}')